# DeepfakeBench Score Extraction

This notebook builds DeepfakeBench-compatible dataset JSON from your frame manifest,
runs detector inference, and exports frame/video-level scores.


## 0. One-time setup on server

Run in terminal (outside notebook) if DeepfakeBench is not prepared yet:

```bash
git clone https://github.com/SCLBD/DeepfakeBench.git
cd DeepfakeBench
# Follow their install instructions (conda + install.sh) or your own env setup.
# Download pretrained detector weights and put them in ./training/weights
```


In [1]:
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader


In [2]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path("..").resolve()
DEEPFAKEBENCH_ROOT = Path("~/deepfake-emotion-robustness/DeepfakeBench").expanduser().resolve()  # change if needed

# Input from your pipeline
SUBSET_NAME = "final"
FRAME_MANIFEST_CSV = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_frame_manifest.csv"

# DeepfakeBench detector config and weights
DETECTOR_CONFIG = DEEPFAKEBENCH_ROOT / "training/config/detector/xception.yaml"
WEIGHTS_PATH = DEEPFAKEBENCH_ROOT / "training/weights/xception_best.pth"

# Custom dataset names/labels used inside DeepfakeBench JSON
CUSTOM_DATASET_NAME = "ThesisFinal"
CUSTOM_REAL_LABEL = "THESIS_real"
CUSTOM_FAKE_LABEL = "THESIS_fake"

# Output
OUT_DIR = PROJECT_ROOT / "outputs/deepfakebench_scores"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_JSON_DIR = OUT_DIR / "dataset_json"
DATASET_JSON_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_JSON_PATH = DATASET_JSON_DIR / f"{CUSTOM_DATASET_NAME}.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEEPFAKEBENCH_ROOT:", DEEPFAKEBENCH_ROOT)
print("FRAME_MANIFEST_CSV:", FRAME_MANIFEST_CSV)
print("DETECTOR_CONFIG:", DETECTOR_CONFIG)
print("WEIGHTS_PATH:", WEIGHTS_PATH)


PROJECT_ROOT: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness
DEEPFAKEBENCH_ROOT: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/DeepfakeBench
FRAME_MANIFEST_CSV: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_frame_manifest.csv
DETECTOR_CONFIG: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/DeepfakeBench/training/config/detector/xception.yaml
WEIGHTS_PATH: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/DeepfakeBench/training/weights/xception_best.pth


In [3]:
assert DEEPFAKEBENCH_ROOT.exists(), f"DeepfakeBench root not found: {DEEPFAKEBENCH_ROOT}"
assert FRAME_MANIFEST_CSV.exists(), f"Frame manifest not found: {FRAME_MANIFEST_CSV}"
assert DETECTOR_CONFIG.exists(), f"Detector config not found: {DETECTOR_CONFIG}"
assert WEIGHTS_PATH.exists(), f"Weights not found: {WEIGHTS_PATH}"


In [4]:
import sys
sys.path.insert(0, "training")

from dataset.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR

print("OK dataset import")
print("Registry type:", type(DETECTOR))
print("xception in registry:", "xception" in DETECTOR.data)
print("Some detectors:", list(DETECTOR.data.keys())[:10])


['in_channels', 'out_channels', 'kernel_size', 'stride', 'padding', 'dilation', 'groups', 'bias', 'padding_mode', 'device', 'dtype']
spatial_count=0 keep_stride_count=0
OK dataset import
Registry type: <class 'metrics.registry.Registry'>
xception in registry: True
Some detectors: ['multi_attention', 'facexray', 'xception', 'efficientnetb4', 'resnet34', 'f3net', 'meso4', 'meso4Inception', 'spsl', 'core']


In [5]:
# Import DeepfakeBench modules
import sys
sys.path.insert(0, str(DEEPFAKEBENCH_ROOT / "training"))

from dataset.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR


In [6]:
# Build runtime config (detector yaml + test config + overrides)
with open(DETECTOR_CONFIG, "r") as f:
    cfg = yaml.safe_load(f)
with open(DEEPFAKEBENCH_ROOT / "training/config/test_config.yaml", "r") as f:
    test_cfg = yaml.safe_load(f)
cfg.update(test_cfg)

cfg["test_dataset"] = [CUSTOM_DATASET_NAME]
cfg["dataset_json_folder"] = str(DATASET_JSON_DIR)
rgb_root = (PROJECT_ROOT / "datasets" / "processed" / SUBSET_NAME / "datasets" / "frames").resolve()
cfg["rgb_dir"] = rgb_root.as_posix()  # No trailing slash - abstract_dataset.py will add separator
cfg["weights_path"] = str(WEIGHTS_PATH)
cfg["lmdb"] = False
cfg["workers"] = int(cfg.get("workers", 4))
cfg["cuda"] = bool(torch.cuda.is_available())
cfg["label_dict"] = {
    CUSTOM_REAL_LABEL: 0,
    CUSTOM_FAKE_LABEL: 1,
}

print("Config loaded. Model:", cfg.get("model_name"))
print("RGB dir:", cfg["rgb_dir"])


Config loaded. Model: xception
RGB dir: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/processed/final/datasets/frames


In [7]:
def normalize_frame_entry(s: str) -> str:
    """Clean and normalize a single frame path entry to preserve full path
    
    Handles: backslashes, .// prefixes, absolute paths, mixed separators.
    Preserves the complete normalized path to ensure exact matching.
    """
    s = str(s).strip()
    
    # Step 1: Normalize all separators to forward slashes
    s = s.replace('\\', '/')
    
    # Step 2: Remove .// and multiple // prefixes
    while s.startswith('.//'):
        s = s[3:]
    while s.startswith('//'):
        s = s[1:]
    s = s.lstrip('./')
    s = s.lstrip('/')
    
    # Step 3: Return the full normalized path (not truncated)
    return s


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# Inference loop (equivalent to test.py, but we keep predictions for CSV export)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_cfg_single = cfg.copy()
test_cfg_single["test_dataset"] = CUSTOM_DATASET_NAME

test_set = DeepfakeAbstractBaseDataset(config=test_cfg_single, mode="test")
test_loader = DataLoader(
    dataset=test_set,
    batch_size=cfg["test_batchSize"],
    shuffle=False,
    num_workers=int(cfg["workers"]),
    collate_fn=test_set.collate_fn,
    drop_last=False,
)

model_class = DETECTOR[cfg["model_name"]]
model = model_class(cfg).to(device)
ckpt = torch.load(cfg["weights_path"], map_location=device)
model.load_state_dict(ckpt, strict=True)
model.eval()

all_probs = []
all_labels = []
stop_reason = None

with torch.no_grad():
    for batch_idx, batch in enumerate(
        tqdm(test_loader, total=len(test_loader), desc="DeepfakeBench inference")
    ):
        try:
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter("always")

                labels = torch.where(batch["label"] != 0, 1, 0)
                batch["image"] = batch["image"].to(device)
                batch["label"] = labels.to(device)

                if batch["mask"] is not None:
                    batch["mask"] = batch["mask"].to(device)
                if batch["landmark"] is not None:
                    batch["landmark"] = batch["landmark"].to(device)

                pred = model(batch, inference=True)
                probs = pred["prob"].detach().cpu().numpy().reshape(-1)
                labs = batch["label"].detach().cpu().numpy().reshape(-1)

                # break if any warning appeared in this batch
                if caught_warnings:
                    first_warning = caught_warnings[0]
                    stop_reason = (
                        f"Stopped on warning at batch {batch_idx}: "
                        f"{first_warning.category.__name__}: {first_warning.message}"
                    )
                    print(stop_reason)
                    break

                all_probs.extend(probs.tolist())
                all_labels.extend(labs.tolist())

        except Exception as e:
            stop_reason = f"Stopped on error at batch {batch_idx}: {type(e).__name__}: {e}"
            print(stop_reason)
            break

frame_paths = test_set.data_dict["image"][: len(all_probs)]
assert len(frame_paths) == len(all_probs), "Mismatch between number of processed frames and scores"

frame_scores_df = pd.DataFrame({
    "frame_path": pd.Series(frame_paths, dtype=str),
    "detector_score": np.array(all_probs, dtype=float),
    "label": np.array(all_labels, dtype=int),
})

# video_id is parent folder name in your preprocessing output
frame_scores_df["video_id"] = frame_scores_df["frame_path"].apply(lambda p: Path(p).parent.name)

video_scores_df = (
    frame_scores_df
    .groupby("video_id", as_index=False)
    .agg(
        detector_score=("detector_score", "mean"),
        n_frames=("frame_path", "count"),
        label=("label", "max"),
    )
)

frame_out = OUT_DIR / f"{CUSTOM_DATASET_NAME}_{cfg['model_name']}_frame_scores.csv"
video_out = OUT_DIR / f"{CUSTOM_DATASET_NAME}_{cfg['model_name']}_video_scores.csv"
frame_scores_df.to_csv(frame_out, index=False)
video_scores_df.to_csv(video_out, index=False)

print("Saved frame scores:", frame_out)
print("Saved video scores:", video_out)
print(video_scores_df.head())

if stop_reason is not None:
    print("\nInference finished early.")
    print(stop_reason)

DeepfakeBench inference: 100%|██████████| 154/154 [00:06<00:00, 25.24it/s]

Saved frame scores: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/outputs/deepfakebench_scores/ThesisFinal_xception_frame_scores.csv
Saved video scores: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/outputs/deepfakebench_scores/ThesisFinal_xception_video_scores.csv
                          video_id  detector_score  n_frames  label
0  Celeb-real__id10_0000__9273e8bc        0.131907        32      0
1  Celeb-real__id10_0002__6a16be4b        0.197790        32      0
2  Celeb-real__id10_0004__2f2a56d2        0.216124        32      0
3  Celeb-real__id10_0006__556711a5        0.286550        29      0
4  Celeb-real__id10_0007__841eee79        0.310433        32      0


: 